In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pingouin import wilcoxon

sns.set_theme(style="whitegrid", font_scale=1.5)

In [2]:
def compute_wilcoxon(df, metric, alternative='greater', approach_name='RFMut'):
    results = pd.DataFrame(columns=['data_attr', 'approach', 'metric', 'pval', 'effect_size'])
    for data_attr in df['data_attr'].unique():
        meg_values = df[(df['data_attr'] == data_attr) & (df['approach'] == approach_name)][metric]
        for approach in df['approach'].unique():
            if approach != approach_name:
                approach_values = df[(df['data_attr'] == data_attr) & (df['approach'] == approach)][metric]
                if not meg_values.empty and not approach_values.empty:
                    stat = wilcoxon(meg_values, approach_values, alternative=alternative)
                    pval = stat['p-val'].values[0]
                    effect_size = stat['CLES'].values[0]
                    results = pd.concat([results, pd.DataFrame({
                        'data_attr': [data_attr],
                        'approach': [approach],
                        'metric': [metric],
                        'pval': [pval],
                        'effect_size': [effect_size]
                    })], ignore_index=True)
    results['pval'] = results['pval'].round(3)
    results['effect_size'] = results['effect_size'].round(3)  
    
    return results

def win_tie_loss(df):
    results = pd.DataFrame()
    for model in df['approach'].unique():
        wins = df[(df['approach'] == model) & (df['pval'] < 0.05)]
        ties = df[(df['approach'] == model) & (df['pval'] >= 0.05) & (df['pval'] <= 0.9)]
        losses = df[(df['approach'] == model) & (df['pval'] > 0.9)]
        results = pd.concat([results, pd.DataFrame({'approach': model, 'wins': wins.shape[0], 'ties': ties.shape[0], 'losses': losses.shape[0]}, index=[0])], ignore_index=True)
    return results

def compute_win_tie_losses(full_data, approach_name='RFMut'):
  full_data['data_attr'] = full_data['data'] + "_" + full_data['attr']
  ris_acc = compute_wilcoxon(full_data, 'Accuracy', alternative='greater', approach_name=approach_name)
  ris_prec = compute_wilcoxon(full_data, 'Precision', alternative='greater', approach_name=approach_name)
  ris_recall = compute_wilcoxon(full_data, 'Recall', alternative='greater', approach_name=approach_name)
  ris_f1 = compute_wilcoxon(full_data, 'F1', alternative='greater', approach_name=approach_name)
  ris_mcc = compute_wilcoxon(full_data, 'MCC', alternative='greater', approach_name=approach_name)
  ris_spd = compute_wilcoxon(full_data, 'SPD', alternative='less', approach_name=approach_name)
  ris_eod = compute_wilcoxon(full_data, 'EOD', alternative='less', approach_name=approach_name)
  ris_aod = compute_wilcoxon(full_data, 'AOD', alternative='less', approach_name=approach_name)
  win_tie_loss_acc = win_tie_loss(ris_acc)
  win_tie_loss_prec = win_tie_loss(ris_prec)
  win_tie_loss_recall = win_tie_loss(ris_recall)
  win_tie_loss_f1 = win_tie_loss(ris_f1)
  win_tie_loss_mcc = win_tie_loss(ris_mcc)
  win_tie_loss_spd = win_tie_loss(ris_spd)
  win_tie_loss_eod = win_tie_loss(ris_eod)
  win_tie_loss_aod = win_tie_loss(ris_aod)
  win_tie_loss_acc['metric'] = 'Accuracy'
  win_tie_loss_prec['metric'] = 'Precision'
  win_tie_loss_recall['metric'] = 'Recall'
  win_tie_loss_f1['metric'] = 'F1'
  win_tie_loss_mcc['metric'] = 'MCC'
  win_tie_loss_spd['metric'] = 'SPD'
  win_tie_loss_eod['metric'] = 'EOD'
  win_tie_loss_aod['metric'] = 'AOD'

  win_tie_losses = pd.concat([
      win_tie_loss_acc,
      win_tie_loss_prec,
      win_tie_loss_recall,
      win_tie_loss_f1,
      win_tie_loss_mcc,
      win_tie_loss_spd,
      win_tie_loss_eod,
      win_tie_loss_aod
  ], ignore_index=True)
  win_tie_losses['win_tie_loss'] = win_tie_losses['wins'].astype(str) + '/' + win_tie_losses['ties'].astype(str) + '/' + win_tie_losses['losses'].astype(str)
  win_tie_losses = win_tie_losses.pivot(index='approach', columns='metric', values=['win_tie_loss']).T
  win_tie_losses.index = win_tie_losses.index.droplevel(0)
  win_tie_losses = win_tie_losses.loc[['Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'SPD', 'EOD', 'AOD']]
  return win_tie_losses

def pareto_front_full(df):
    df = df.sort_values(by=['Accuracy', 'Precision', 'Recall', 'MCC', 'SPD', 'EOD', 'AOD'], ascending=[False, False, False, False, True, True, True])
    pareto = []
    for i in range(len(df)):
        if not any((df.iloc[i]['Accuracy'] <= p['Accuracy'] and
                    df.iloc[i]['Precision'] <= p['Precision'] and
                    df.iloc[i]['Recall'] <= p['Recall'] and
                    df.iloc[i]['MCC'] <= p['MCC'] and
                    df.iloc[i]['SPD'] >= p['SPD'] and
                    df.iloc[i]['EOD'] >= p['EOD'] and
                    df.iloc[i]['AOD'] >= p['AOD']) for p in pareto):
            pareto.append(df.iloc[i])
    return pd.DataFrame(pareto)

def pareto_front_reduced(df):
    df = df.sort_values(by=['MCC', 'SPD'], ascending=[False, True])
    pareto = []
    for i in range(len(df)):
        if not any((df.iloc[i]['MCC'] <= p['MCC'] and
                    df.iloc[i]['SPD'] >= p['SPD']) for p in pareto):
            pareto.append(df.iloc[i])
    return pd.DataFrame(pareto)

In [3]:
def read_results(file_path, strategy):
    meg_ris = pd.DataFrame()
    for file in os.listdir(file_path):
        data = file.split("_")[0]
        attr = file.split("_")[1]
        df = pd.read_csv(f"{file_path}/{file}", index_col=0)
        df["data"] = data
        df["attr"] = attr
        meg_ris = pd.concat([meg_ris, df])
        meg_ris["approach"] = strategy
    return meg_ris

# RF Mut vs Base Classifiers

In [4]:
rf_mut_ris = read_results("results_rf", "FairRF")

In [5]:
rf_mean = rf_mut_ris.groupby(['data', 'attr', 'approach','Run']).mean().reset_index()

In [6]:
rs = read_results("results_rs", "RS")

In [7]:
rs_mean = rs.groupby(['data', 'attr', 'approach','Run']).mean().reset_index()

In [8]:
baseline = pd.read_csv("baselines_results.csv")

In [9]:
baseline['SPD'] = baseline['SPD'].abs()
baseline['EOD'] = baseline['EOD'].abs()
baseline['AOD'] = baseline['AOD'].abs()

In [10]:
full_data = pd.concat([baseline, rs_mean, rf_mean], ignore_index=True)

In [11]:
merged_data = full_data.groupby(['data', 'attr', 'approach']).aggregate(['mean', 'std']).reset_index().round(3)

In [12]:
merged_data['Acc'] = merged_data['Accuracy']['mean'].astype(str) + "$\pm$" + merged_data['Accuracy']['std'].astype(str)
merged_data['Prec'] = merged_data['Precision']['mean'].astype(str) + "$\pm$" + merged_data['Precision']['std'].astype(str)
merged_data['Rec'] = merged_data['Recall']['mean'].astype(str) + "$\pm$" + merged_data['Recall']['std'].astype(str)
merged_data['F1Score'] = merged_data['F1']['mean'].astype(str) + "$\pm$" + merged_data['F1']['std'].astype(str)
merged_data['mcc'] = merged_data['MCC']['mean'].astype(str) + "$\pm$" + merged_data['MCC']['std'].astype(str)
merged_data['SP'] = merged_data['SPD']['mean'].astype(str) + "$\pm$" + merged_data['SPD']['std'].astype(str)
merged_data['EO'] = merged_data['EOD']['mean'].astype(str) + "$\pm$" + merged_data['EOD']['std'].astype(str)
merged_data['AO'] = merged_data['AOD']['mean'].astype(str) + "$\pm$" + merged_data['AOD']['std'].astype(str)

In [13]:
merged_data.drop(columns=['Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'SPD', 'EOD', 'AOD'], inplace=True)

/var/folders/4c/mgvn0dc97_9gst9l7jbv9n640000gn/T/ipykernel_68488/3077891099.py:1: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  merged_data.drop(columns=['Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'SPD', 'EOD', 'AOD'], inplace=True)


In [14]:
merged_data.columns = merged_data.columns.droplevel(1)

In [15]:
merged_data

,data,attr,approach,Run,Run,Acc,Prec,Rec,F1Score,mcc,SP,EO,AO
0,adult,race,FairRF,9.5,5.916,0.855$\pm$0.003,0.829$\pm$0.003,0.76$\pm$0.005,0.784$\pm$0.005,0.585$\pm$0.008,0.073$\pm$0.005,0.032$\pm$0.015,0.024$\pm$0.01
1,adult,race,RS,9.5,5.916,0.844$\pm$0.005,0.8$\pm$0.009,0.767$\pm$0.006,0.781$\pm$0.006,0.566$\pm$0.012,0.09$\pm$0.006,0.047$\pm$0.015,0.04$\pm$0.01
2,adult,race,forest,NaN,NaN,0.841$\pm$0.002,0.836$\pm$0.002,0.841$\pm$0.002,0.838$\pm$0.002,0.559$\pm$0.005,0.102$\pm$0.006,0.049$\pm$0.024,0.047$\pm$0.015
3,adult,race,knn,NaN,NaN,0.82$\pm$0.003,0.813$\pm$0.003,0.82$\pm$0.003,0.816$\pm$0.003,0.498$\pm$0.009,0.099$\pm$0.007,0.091$\pm$0.029,0.067$\pm$0.016
4,adult,race,logreg,NaN,NaN,0.846$\pm$0.003,0.839$\pm$0.003,0.846$\pm$0.003,0.84$\pm$0.003,0.564$\pm$0.007,0.099$\pm$0.007,0.082$\pm$0.032,0.06$\pm$0.018
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,park,sex,forest,NaN,NaN,0.696$\pm$0.009,0.702$\pm$0.009,0.696$\pm$0.009,0.693$\pm$0.01,0.524$\pm$0.014,0.086$\pm$0.023,0.05$\pm$0.03,0.045$\pm$0.025
87,park,sex,knn,NaN,NaN,0.671$\pm$0.01,0.671$\pm$0.01,0.671$\pm$0.01,0.669$\pm$0.01,0.486$\pm$0.016,0.075$\pm$0.021,0.025$\pm$0.02,0.032$\pm$0.017
88,park,sex,logreg,NaN,NaN,0.504$\pm$0.01,0.463$\pm$0.023,0.504$\pm$0.01,0.442$\pm$0.016,0.191$\pm$0.012,0.215$\pm$0.028,0.249$\pm$0.047,0.207$\pm$0.033
89,park,sex,svm,NaN,NaN,0.614$\pm$0.009,0.621$\pm$0.009,0.614$\pm$0.009,0.611$\pm$0.01,0.396$\pm$0.014,0.238$\pm$0.02,0.256$\pm$0.034,0.22$\pm$0.022


In [16]:
bin_data = full_data[full_data['data'].isin([
    'adult',
    'bank',
    'compas',
    'german',
    'mep',
])]

In [22]:
grouped = bin_data[['SPD', 'EOD', 'AOD', 'Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'approach']].groupby('approach').mean()

In [24]:
# Compute percentage difference in SPD, EOD and AOD between FairRF and other approaches
fairrf_means = grouped.loc['FairRF']
percentage_diff = pd.DataFrame()

for approach in grouped.index:
  if approach != 'FairRF':
    spd_diff = ((grouped.loc[approach, 'SPD'] - fairrf_means['SPD']) / fairrf_means['SPD']) * 100
    eod_diff = ((grouped.loc[approach, 'EOD'] - fairrf_means['EOD']) / fairrf_means['EOD']) * 100
    aod_diff = ((grouped.loc[approach, 'AOD'] - fairrf_means['AOD']) / fairrf_means['AOD']) * 100

    acc_diff = ((grouped.loc[approach, 'Accuracy'] - fairrf_means['Accuracy']) / fairrf_means['Accuracy']) * 100
    prec_diff = ((grouped.loc[approach, 'Precision'] - fairrf_means['Precision']) / fairrf_means['Precision']) * 100
    recall_diff = ((grouped.loc[approach, 'Recall'] - fairrf_means['Recall']) / fairrf_means['Recall']) * 100
    f1_diff = ((grouped.loc[approach, 'F1'] - fairrf_means['F1']) / fairrf_means['F1']) * 100
    mcc_diff = ((grouped.loc[approach, 'MCC'] - fairrf_means['MCC']) / fairrf_means['MCC']) * 100
    
    percentage_diff = pd.concat([percentage_diff, pd.DataFrame({
      'approach': [approach],
      'SPD_diff_%': [spd_diff],
      'EOD_diff_%': [eod_diff],
      'AOD_diff_%': [aod_diff],
      'Accuracy_diff_%': [acc_diff],
      'Precision_diff_%': [prec_diff],
      'Recall_diff_%': [recall_diff],
      'F1_diff_%': [f1_diff],
      'MCC_diff_%': [mcc_diff]
    })], ignore_index=True)

percentage_diff = percentage_diff.round(2)
percentage_diff

,approach,SPD_diff_%,EOD_diff_%,AOD_diff_%,Accuracy_diff_%,Precision_diff_%,Recall_diff_%,F1_diff_%,MCC_diff_%
0,RS,18.15,21.75,23.54,-1.35,-2.98,0.05,-0.57,-4.49
1,forest,31.75,31.93,40.33,-1.04,3.09,13.42,9.60,-2.89
2,knn,38.45,79.80,76.50,-3.80,-0.47,10.25,6.02,-20.81
3,logreg,63.42,114.93,104.37,-0.25,3.89,14.32,10.57,-0.20
4,svm,50.45,96.82,100.73,-1.31,2.58,13.11,8.33,-9.84
5,tree,21.75,10.33,22.47,-6.75,-1.82,6.88,4.47,-22.97


### Pareto Optimality

In [56]:
from itertools import product

datasets = ["adult","compas","german","mep"]
attrs = ["age","race","sex"]
for dataset, attr in product(datasets, attrs):
    #filtered = grouped[(grouped["data"]==dataset) & (grouped["attr"]==attr)]
    filtered = full_data[(full_data["data"]==dataset) & (full_data["attr"]==attr)]
    if not len(filtered):
        continue
    pareto = pareto_front_reduced(filtered)
    print (dataset, attr)
    print (pareto['approach'].value_counts())
    print ("----------")

adult race
approach
FairRF    4
Name: count, dtype: int64
----------
adult sex
approach
FairRF    5
Name: count, dtype: int64
----------
compas race
approach
tree      4
FairRF    2
logreg    1
RS        1
knn       1
Name: count, dtype: int64
----------
compas sex
approach
FairRF    3
logreg    1
Name: count, dtype: int64
----------
german age
approach
FairRF    3
logreg    1
RS        1
forest    1
svm       1
Name: count, dtype: int64
----------
german sex
approach
logreg    2
FairRF    1
knn       1
forest    1
Name: count, dtype: int64
----------
